# BioTek EL406

The BioTek EL406 plate washer has four subsystems — manifold, syringe pump, peristaltic pump, and shaker — each exposed as a capability on the `EL406` device.

## Setup

In [1]:
import os

from pylabrobot.agilent.biotek.el406 import EL406
from pylabrobot.resources import Cor_96_wellplate_360ul_Fb

os.environ["DYLD_LIBRARY_PATH"] = "/opt/homebrew/lib:" + os.environ.get("DYLD_LIBRARY_PATH", "")
el406 = EL406(name="el406")
await el406.setup()

plate = Cor_96_wellplate_360ul_Fb(name="plate")

2026-03-31 17:04:26,351 - pylabrobot.agilent.biotek.el406.driver - INFO - EL406Driver setting up
2026-03-31 17:04:26,352 - pylabrobot.agilent.biotek.el406.driver - INFO -   Timeout: 15.0 seconds
2026-03-31 17:04:26,435 - pylabrobot.io.ftdi - INFO - Successfully opened FTDI device: 1504309
2026-03-31 17:04:26,439 - pylabrobot.agilent.biotek.el406.driver - INFO -   Serial: 38400 baud, 8N2
2026-03-31 17:04:26,441 - pylabrobot.agilent.biotek.el406.driver - INFO -   Flow control: NONE
2026-03-31 17:04:26,459 - pylabrobot.agilent.biotek.el406.driver - INFO - Testing communication with device...
2026-03-31 17:04:26,489 - pylabrobot.agilent.biotek.el406.communication - INFO - EL406 communication test passed
2026-03-31 17:04:26,489 - pylabrobot.agilent.biotek.el406.communication - INFO - Sending INIT_STATE command (0xA0) to clear device state
2026-03-31 17:04:26,616 - pylabrobot.agilent.biotek.el406.driver - INFO -   Communication test: PASSED
2026-03-31 17:04:26,616 - pylabrobot.agilent.biotek

2026-03-31 17:04:41,188 - pylabrobot.agilent.biotek.el406.actions - INFO - Instrument reset complete
2026-03-31 17:04:41,189 - pylabrobot.agilent.biotek.el406.driver - INFO -   Instrument reset: DONE
2026-03-31 17:04:41,190 - pylabrobot.agilent.biotek.el406.driver - INFO - EL406Driver setup complete


## Manifold (plate washing)

The wash manifold is the primary fluid system.

In [ ]:
from pylabrobot.agilent.biotek.el406.plate_washing_backend import EL406PlateWashingBackend

# Basic wash
await el406.washer.wash(plate, cycles=3, dispense_volume=300)

# Wash with buffer and shake/soak
await el406.washer.wash(
    plate,
    cycles=3,
    dispense_volume=300,
    backend_params=EL406PlateWashingBackend.WashParams(
        buffer="A",
        soak_duration=10,
        shake_duration=5,
        shake_intensity="Medium",
    ),
)

# Aspirate and dispense
await el406.washer.aspirate(plate)
await el406.washer.dispense(plate, volume=200)

# Prime
await el406.washer.prime(
    backend_params=EL406PlateWashingBackend.PrimeParams(
        plate=plate, volume=10000, buffer="A",
    ),
)

## Syringe pump

Precise low-volume dispensing via dual syringe pumps (A/B).

In [ ]:
from pylabrobot.agilent.biotek.el406.syringe_dispensing_backend import EL406SyringeDispensingBackend

await el406.syringe.prime(plate, volume=5000, backend_params=EL406SyringeDispensingBackend.PrimeParams(syringe="A"))

# Same volume for all columns
await el406.syringe.dispense(plate, volumes={c: 50 for c in range(1, 13)})

Different volumes per column:

In [ ]:
# Different volumes per column: columns 1-6 get 100 uL, columns 7-12 get 50 uL
await el406.syringe.dispense(plate, volumes={
    **{c: 100 for c in range(1, 7)},
    **{c: 50 for c in range(7, 13)},
})

## Peristaltic pump

Continuous-flow dispensing with cassette selection and row/column masking.

In [ ]:
from pylabrobot.agilent.biotek.el406.peristaltic_dispensing_backend import EL406PeristalticDispensingBackend

await el406.peristaltic.prime(plate, volume=300, backend_params=EL406PeristalticDispensingBackend.PrimeParams(flow_rate="High"))

# Same volume for all columns
await el406.peristaltic.dispense(plate, volumes={c: 100 for c in range(1, 13)})

# Different volumes per column
await el406.peristaltic.dispense(plate, volumes={
    **{c: 200 for c in range(1, 7)},
    **{c: 100 for c in range(7, 13)},
})

await el406.peristaltic.purge(plate, volume=300, backend_params=EL406PeristalticDispensingBackend.PrimeParams(flow_rate="High"))

## Shaking

The EL406 shake is a single fire-and-forget command with duration baked in, so it uses the backend API directly rather than the generic `Shaker.shake(speed, duration)` interface.

In [ ]:
await el406.shaker.backend.shake(plate, duration=30, intensity="Medium")

## Device-level operations

In [ ]:
from pylabrobot.agilent.biotek.el406.enums import EL406WasherManifold

await el406._driver.set_washer_manifold(EL406WasherManifold.TUBE_96_DUAL)
await el406._driver.reset()

## Teardown

In [ ]:
await el406.stop()